# 09 - Desigualdad territorial ENIGH 2018-2024

Este notebook resume una etapa descriptiva de desigualdad territorial para la tesina. El objetivo es calcular Gini ponderado, compararlo con el benchmark de Banco de México cuando la definición sea compatible, y separar desigualdad dentro de territorios de brechas entre territorios.

No se crean modelos ni se deflacta. Se incorpora solo una fuente oficial acotada para zonas metropolitanas y no se aproximan grandes urbes con municipios sueltos.

## Definición metodológica

Benchmark revisado: Banco de México, Recuadro 2 del Reporte sobre las Economías Regionales enero-marzo 2024, "Disminución de la desigualdad de ingresos regional en un contexto de crecimiento propobre: 2018-2022".

Punto clave: el recuadro indica que el Gini usa **ingreso corriente total promedio por hogar**. Por eso la comparación principal de este notebook usa:

- ingreso: `ing_cor_hogar_oficial_tri`;
- ponderador: `factor`;
- universo: todos los hogares con ingreso no faltante, no negativo y factor positivo;
- ceros: se conservan como valores válidos;
- escala: se reporta Gini en 0-1 y 0-100;
- años comparables con Banxico: 2018, 2020 y 2022.

La comparación no busca calzar perfectamente. Banco de México usa bases generadas por CONEVAL a partir de ENIGH; aquí se usa el mart propio con la variable oficial de `concentradohogar`. Diferencias pequeñas o moderadas pueden venir de definición CONEVAL, procesamiento, universo, ponderación, escala temporal del ingreso o redondeo.

In [26]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (10, 5), "axes.titlesize": 13, "axes.labelsize": 11})

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "interim" / "revision_4").exists():
            return candidate
    raise FileNotFoundError("No se encontró data/interim/revision_4 desde el directorio actual.")

ROOT = find_project_root()
REV4 = ROOT / "data" / "interim" / "revision_4"
HOGAR_PATH = REV4 / "mart_hogar_2018_2024.csv.gz"
PERSONA_PATH = REV4 / "mart_persona_2018_2024.csv.gz"

WEIGHT = "factor"
INC_HH = "ing_cor_hogar_oficial_tri"
INC_PC = "ing_cor_pc_oficial_tri"
INC_PC_PERSONA = "ing_cor_hogar_pc_oficial_tri"
INC_LAB = "ingreso_persona_laboral_negocio_tri"
W_PERSONAS_DESDE_HOGAR = "peso_personas_desde_hogar"
YEARS = [2018, 2020, 2022, 2024]

print("Rutas relativas detectadas:")
print(f"Base hogares: {HOGAR_PATH.exists()} | {HOGAR_PATH.relative_to(ROOT).as_posix()}")
print(f"Base personas: {PERSONA_PATH.exists()} | {PERSONA_PATH.relative_to(ROOT).as_posix()}")

Rutas relativas detectadas:
Base hogares: True | data/interim/revision_4/mart_hogar_2018_2024.csv.gz
Base personas: True | data/interim/revision_4/mart_persona_2018_2024.csv.gz


## Figuras documentales y llaves

Las figuras que se insertan en la documentación se guardan en `reports/figures_documentacion/`, una carpeta específica y versionable. No se usan rutas absolutas en el notebook ni en el Markdown.


In [2]:
RAW = ROOT / "data" / "raw" / "EINGH"
DOC_FIG_DIR = ROOT / "reports" / "figures_documentacion"
DOC_FIG_DIR.mkdir(parents=True, exist_ok=True)
ZM_MAP_PATH = ROOT / "docs" / "zonas_metropolitanas_prioritarias_2020.csv"


def save_doc_figure(fig, filename, alt_text):
    """Guarda figura versionable y la muestra en el notebook por ruta relativa."""
    path = DOC_FIG_DIR / filename
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    display(Markdown(f"![{alt_text}](../reports/figures_documentacion/{filename})"))
    return path


def as_code(series, width):
    return pd.to_numeric(series, errors="coerce").astype("Int64").astype("string").str.zfill(width)


## Carga de bases analíticas

Se cargan únicamente las columnas necesarias para esta revisión. La base de hogares se usa para Gini comparable con Banxico y para estimandos cuya unidad es el hogar. La base de personas se usa para ingreso laboral individual y, desde esta auditoría, como fuente principal del ingreso corriente per cápita cuando el estimando es la distribución **entre personas**.

In [3]:
hogar_cols = [
    "anio", "folioviv", "foliohog", "region_banxico", "entidad", "cve_ent", "tam_loc_desc", "est_socio_desc",
    WEIGHT, "factor_hogar", "est_dis", "upm", INC_HH, INC_PC, "ingtrab_hogar_oficial_tri", "tot_integ",
]
persona_cols = [
    "anio", "folioviv", "foliohog", "numren", "region_banxico", "entidad", "cve_ent", "tam_loc_desc", "est_socio_desc",
    "sexo_desc", WEIGHT, INC_LAB, INC_PC_PERSONA, "edad",
]
hogar = pd.read_csv(HOGAR_PATH, usecols=hogar_cols, low_memory=False)
persona = pd.read_csv(PERSONA_PATH, usecols=persona_cols, low_memory=False)

for col in ["anio", WEIGHT, "factor_hogar", INC_HH, INC_PC, "ingtrab_hogar_oficial_tri", "tot_integ"]:
    hogar[col] = pd.to_numeric(hogar[col], errors="coerce")
for col in ["anio", WEIGHT, INC_LAB, INC_PC_PERSONA, "edad"]:
    persona[col] = pd.to_numeric(persona[col], errors="coerce")

hogar[W_PERSONAS_DESDE_HOGAR] = hogar[WEIGHT] * hogar["tot_integ"]
persona["peso_persona"] = persona[WEIGHT]

validacion_base = pd.DataFrame([
    {"base": "hogar", "filas": len(hogar), "columnas": hogar.shape[1], "factor_missing": int(hogar[WEIGHT].isna().sum()), "ingreso_missing": int(hogar[INC_HH].isna().sum()), "ingreso_negativo": int(hogar[INC_HH].lt(0).sum())},
    {"base": "persona", "filas": len(persona), "columnas": persona.shape[1], "factor_missing": int(persona[WEIGHT].isna().sum()), "ingreso_laboral_missing": int(persona[INC_LAB].isna().sum()), "ingreso_laboral_negativo": int(persona[INC_LAB].lt(0).sum())},
])
display(validacion_base)

,base,filas,columnas,factor_missing,ingreso_missing,ingreso_negativo,ingreso_laboral_missing,ingreso_laboral_negativo
0,hogar,345169,17,0,0.0,0.0,NaN,NaN
1,persona,1203231,15,0,NaN,NaN,0.0,0.0


## Funciones ponderadas

La fórmula operativa del Gini usa la curva de Lorenz ponderada. Se ordena el ingreso de menor a mayor, se acumula población ponderada e ingreso ponderado, y se calcula:

```text
Gini = 1 - 2 * área bajo la curva de Lorenz
```

El cálculo conserva ceros y excluye únicamente ingresos faltantes, negativos o pesos no positivos.

In [4]:
def weighted_quantile(values, weights, qs):
    x = pd.to_numeric(values, errors="coerce").to_numpy(dtype="float64")
    w = pd.to_numeric(weights, errors="coerce").to_numpy(dtype="float64")
    q = np.atleast_1d(qs).astype(float)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[mask], w[mask]
    if len(x) == 0:
        return np.full(len(q), np.nan)
    order = np.argsort(x, kind="mergesort")
    x, w = x[order], w[order]
    cum_w = np.cumsum(w) / np.sum(w)
    return np.interp(q, cum_w, x, left=x[0], right=x[-1])

def weighted_gini(values, weights):
    x = pd.to_numeric(values, errors="coerce").to_numpy(dtype="float64")
    w = pd.to_numeric(weights, errors="coerce").to_numpy(dtype="float64")
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0) & (x >= 0)
    x, w = x[mask], w[mask]
    if len(x) == 0 or np.sum(x * w) <= 0:
        return np.nan
    order = np.argsort(x, kind="mergesort")
    x, w = x[order], w[order]
    cum_w = np.cumsum(w)
    cum_xw = np.cumsum(x * w)
    pop_share = np.insert(cum_w / cum_w[-1], 0, 0)
    income_share = np.insert(cum_xw / cum_xw[-1], 0, 0)
    return float(1 - 2 * np.trapezoid(income_share, pop_share))

def weighted_summary(df, group_cols, income_col, positive_only=False):
    work = df[df[income_col].gt(0)].copy() if positive_only else df.copy()
    rows = []
    for key, g in work.groupby(group_cols, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        row = dict(zip(group_cols, key))
        x = pd.to_numeric(g[income_col], errors="coerce")
        w = pd.to_numeric(g[WEIGHT], errors="coerce")
        mask = x.notna() & w.notna() & w.gt(0) & x.ge(0)
        x, w = x[mask], w[mask]
        qs = weighted_quantile(x, w, [0.25, 0.50, 0.75, 0.90, 0.95])
        row.update({
            "n": int(mask.sum()),
            "n_ponderado": float(w.sum()),
            "media": float(np.average(x, weights=w)) if len(x) else np.nan,
            "p25": qs[0],
            "mediana": qs[1],
            "p75": qs[2],
            "p90": qs[3],
            "p95": qs[4],
            "gini": weighted_gini(x, w),
        })
        rows.append(row)
    out = pd.DataFrame(rows)
    out["gini_0_100"] = out["gini"] * 100
    return out

def money_cols(df):
    cols = ["media", "p25", "mediana", "p75", "p90", "p95", "gini", "gini_0_100"]
    return df.round({c: 2 for c in cols if c in df.columns})

## Auditoría de `factor`

La documentación oficial de ENIGH define `factor` como el factor de expansión para generalizar resultados de la muestra a la población. La regla metodológica no depende solo del nombre de la variable, sino de la **unidad de observación** de la tabla usada.

Evidencia oficial revisada:

- ENIGH 2018 y 2020: el factor de expansión para vivienda, hogar o persona se encuentra en `factor` de VIVIENDAS y en CONCENTRADOHOGAR. En esas ediciones `poblacion.csv` no trae `factor` directamente, por lo que el mart de personas lo hereda por llaves del hogar/vivienda.
- ENIGH 2022 y 2024: el factor de expansión para cualquier nivel se encuentra en `factor` de VIVIENDAS, HOGARES, POBLACION, GASTOSHOGAR, GASTOSPERSONA, INGRESOS, TRABAJOS y CONCENTRADOHOGAR.
- En filas persona el ponderador correcto es `factor`. No se debe multiplicar otra vez por `tot_integ`.
- `factor * tot_integ` solo es apropiado cuando se parte de una fila por hogar y se quiere representar a las personas integrantes de ese hogar.

In [5]:
factor_evidencia_oficial = pd.DataFrame([
    {"anio": 2018, "fuente_oficial": "data/raw/EINGH/2018/doc_2018.pdf, sección 1.3.3", "regla_documentada": "factor para cualquier nivel en VIVIENDAS y CONCENTRADOHOGAR", "factor_en_poblacion_raw": "No"},
    {"anio": 2020, "fuente_oficial": "data/raw/EINGH/2020/doc_2020.pdf, sección 1.3.3", "regla_documentada": "factor para cualquier nivel en VIVIENDAS y CONCENTRADOHOGAR", "factor_en_poblacion_raw": "No"},
    {"anio": 2022, "fuente_oficial": "data/raw/EINGH/2022/doc_2022.pdf, sección 1.3.3", "regla_documentada": "factor para cualquier nivel en VIVIENDAS, HOGARES, POBLACION y tablas derivadas", "factor_en_poblacion_raw": "Sí"},
    {"anio": 2024, "fuente_oficial": "data/raw/EINGH/2024/doc_2024.pdf, sección 1.3.3", "regla_documentada": "factor para cualquier nivel en VIVIENDAS, HOGARES, POBLACION y tablas derivadas", "factor_en_poblacion_raw": "Sí"},
])
display(factor_evidencia_oficial)

metadata = pd.read_csv(ROOT / "docs" / "enigh_variable_metadata.csv")
year_col = "anio" if "anio" in metadata.columns else "year"
table_col = "tabla" if "tabla" in metadata.columns else "table"
factor_metadata = metadata[metadata["variable"].eq("factor")].copy()
factor_disponibilidad = (
    factor_metadata.assign(disponible="Sí")
    .pivot_table(index=table_col, columns=year_col, values="disponible", aggfunc="first", fill_value="No")
    .reset_index()
    .rename(columns={table_col: "tabla"})
)
display(factor_disponibilidad)

factor_resumen_hogar = hogar.groupby("anio").agg(
    n_hogares=(WEIGHT, "size"),
    hogares_expandidos=(WEIGHT, "sum"),
    poblacion_desde_hogares_integrantes=(W_PERSONAS_DESDE_HOGAR, "sum"),
    factor_min=(WEIGHT, "min"),
    factor_mediana=(WEIGHT, "median"),
    factor_max=(WEIGHT, "max"),
).reset_index()
factor_resumen_persona = persona.groupby("anio").agg(
    n_personas=(WEIGHT, "size"),
    poblacion_desde_personas_factor=(WEIGHT, "sum"),
).reset_index()
factor_resumen = factor_resumen_hogar.merge(factor_resumen_persona, on="anio")
factor_resumen["dif_personas_vs_hogar_integrantes"] = factor_resumen["poblacion_desde_personas_factor"] - factor_resumen["poblacion_desde_hogares_integrantes"]
display(factor_resumen.round(2))

,anio,fuente_oficial,regla_documentada,factor_en_poblacion_raw
0,2018,"data/raw/EINGH/2018/doc_2018.pdf, sección 1.3.3",factor para cualquier nivel en VIVIENDAS y CON...,No
1,2020,"data/raw/EINGH/2020/doc_2020.pdf, sección 1.3.3",factor para cualquier nivel en VIVIENDAS y CON...,No
2,2022,"data/raw/EINGH/2022/doc_2022.pdf, sección 1.3.3","factor para cualquier nivel en VIVIENDAS, HOGA...",Sí
3,2024,"data/raw/EINGH/2024/doc_2024.pdf, sección 1.3.3","factor para cualquier nivel en VIVIENDAS, HOGA...",Sí


year,tabla,2018,2020,2022,2024
0,concentradohogar.csv,Sí,Sí,Sí,Sí
1,gastoshogar.csv,No,No,Sí,Sí
2,gastospersona.csv,No,No,Sí,Sí
3,hogares.csv,No,No,Sí,Sí
4,ingresos.csv,No,No,Sí,Sí
5,poblacion.csv,No,No,Sí,Sí
6,trabajos.csv,No,No,Sí,Sí
7,viviendas.csv,Sí,Sí,Sí,Sí


,anio,n_hogares,hogares_expandidos,poblacion_desde_hogares_integrantes,factor_min,factor_mediana,factor_max,n_personas,poblacion_desde_personas_factor,dif_personas_vs_hogar_integrantes
0,2018,74647,34400515,123836081,9,323.0,6414,269206,123934029,97948
1,2020,89006,35749659,126760856,10,279.0,5284,315743,126838467,77611
2,2022,90102,37560123,128889708,6,283.0,6470,309684,128999038,109330
3,2024,91414,38830230,130226218,4,288.0,7127,308598,130325969,99751


### Contraste de `factor` contra tablas originales

Cuando `factor` existe en las tablas raw, se contrasta contra `concentradohogar` por las llaves disponibles. En 2018 y 2020 varias tablas hijas no lo contienen directamente; en 2022 y 2024 sí aparece repetido y coincide con el factor del hogar.


In [6]:
def audit_factor_across_raw_tables(raw_root, years):
    tables = ["viviendas", "hogares", "poblacion", "trabajos", "ingresos", "gastoshogar", "gastospersona", "concentradohogar"]
    rows = []
    for year in years:
        conc = pd.read_csv(raw_root / str(year) / "concentradohogar.csv", usecols=["folioviv", "foliohog", "factor"], low_memory=False)
        conc["folioviv"] = conc["folioviv"].astype(str)
        conc["foliohog"] = conc["foliohog"].astype(str)
        conc["factor"] = pd.to_numeric(conc["factor"], errors="coerce")
        conc_viv = conc[["folioviv", "factor"]].drop_duplicates()
        for table in tables:
            path = raw_root / str(year) / f"{table}.csv"
            cols = pd.read_csv(path, nrows=0, low_memory=False).columns.tolist()
            if "factor" not in cols:
                rows.append({"anio": year, "tabla": table, "factor_en_tabla": "No", "n_filas": np.nan, "mismatches_con_concentrado": np.nan, "max_dif_factor": np.nan})
                continue
            usecols = ["folioviv", "factor"] + (["foliohog"] if "foliohog" in cols else [])
            df = pd.read_csv(path, usecols=usecols, low_memory=False)
            df["folioviv"] = df["folioviv"].astype(str)
            df["factor"] = pd.to_numeric(df["factor"], errors="coerce")
            if "foliohog" in df.columns:
                df["foliohog"] = df["foliohog"].astype(str)
                merged = df.merge(conc, on=["folioviv", "foliohog"], how="left", suffixes=("_tabla", "_concentrado"))
            else:
                merged = df.merge(conc_viv, on="folioviv", how="left", suffixes=("_tabla", "_concentrado"))
            diff = (merged["factor_tabla"] - merged["factor_concentrado"]).abs()
            rows.append({"anio": year, "tabla": table, "factor_en_tabla": "Sí", "n_filas": len(df), "mismatches_con_concentrado": int((diff.fillna(0) > 0).sum()), "max_dif_factor": float(diff.max()) if diff.notna().any() else np.nan})
    return pd.DataFrame(rows)

factor_tablas = audit_factor_across_raw_tables(RAW, YEARS)
display(factor_tablas)
display(factor_tablas.groupby(["anio", "factor_en_tabla"], dropna=False).agg(tablas=("tabla", "count"), mismatches=("mismatches_con_concentrado", "sum")).reset_index())

poblacion_raw_factor = []
for year in YEARS:
    path = RAW / str(year) / "poblacion.csv"
    cols = pd.read_csv(path, nrows=0, low_memory=False).columns.tolist()
    if "factor" in cols:
        pop_factor = pd.read_csv(path, usecols=["factor"], low_memory=False)["factor"]
        poblacion_raw_factor.append({"anio": year, "factor_directo_en_poblacion_raw": "Sí", "suma_factor_poblacion_raw": pd.to_numeric(pop_factor, errors="coerce").sum()})
    else:
        poblacion_raw_factor.append({"anio": year, "factor_directo_en_poblacion_raw": "No", "suma_factor_poblacion_raw": np.nan})
display(pd.DataFrame(poblacion_raw_factor))

,anio,tabla,factor_en_tabla,n_filas,mismatches_con_concentrado,max_dif_factor
0,2018,viviendas,Sí,73405.0,0.0,0.0
1,2018,hogares,No,NaN,NaN,NaN
2,2018,poblacion,No,NaN,NaN,NaN
3,2018,trabajos,No,NaN,NaN,NaN
4,2018,ingresos,No,NaN,NaN,NaN
5,2018,gastoshogar,No,NaN,NaN,NaN
6,2018,gastospersona,No,NaN,NaN,NaN
7,2018,concentradohogar,Sí,74647.0,0.0,0.0
8,2020,viviendas,Sí,87754.0,0.0,0.0
9,2020,hogares,No,NaN,NaN,NaN


,anio,factor_en_tabla,tablas,mismatches
0,2018,No,6,0.0
1,2018,Sí,2,0.0
2,2020,No,6,0.0
3,2020,Sí,2,0.0
4,2022,Sí,8,0.0
5,2024,Sí,8,0.0


,anio,factor_directo_en_poblacion_raw,suma_factor_poblacion_raw
0,2018,No,NaN
1,2020,No,NaN
2,2022,Sí,128999038.0
3,2024,Sí,130325969.0


## Tabla compacta de auditoría de ponderaciones

Esta tabla resume cómo se deben leer los principales resultados del proyecto. Evita vender como poblacional un resultado muestral.


In [7]:
auditoria_ponderaciones = pd.DataFrame([
    {"Resultado": "Gini nacional y regional", "Unidad observación": "Hogar", "Variable": INC_HH, "Ponderado": "Sí", "Peso utilizado": "factor", "Clasificación": "correcto", "Observación": "Estimación puntual ponderada de hogares; benchmark Banxico no afectado."},
    {"Resultado": "Media, mediana y cuantiles de ingreso corriente del hogar", "Unidad observación": "Hogar", "Variable": INC_HH, "Ponderado": "Sí", "Peso utilizado": "factor", "Clasificación": "correcto", "Observación": "Montos nominales trimestrales; no deflactado."},
    {"Resultado": "Ingreso corriente per cápita entre hogares", "Unidad observación": "Hogar", "Variable": INC_PC, "Ponderado": "Sí", "Peso utilizado": "factor", "Clasificación": "correcto si el estimando son hogares", "Observación": "Pregunta: cómo se distribuye el ingreso per cápita entre hogares."},
    {"Resultado": "Ingreso corriente per cápita entre personas", "Unidad observación": "Persona", "Variable": INC_PC_PERSONA, "Ponderado": "Sí", "Peso utilizado": "factor", "Clasificación": "estimando principal", "Observación": "Cada fila persona hereda el ingreso per cápita del hogar; no multiplicar por tot_integ."},
    {"Resultado": "Ingreso corriente per cápita entre personas desde hogares", "Unidad observación": "Hogar", "Variable": INC_PC, "Ponderado": "Sí", "Peso utilizado": "factor * tot_integ", "Clasificación": "correcto para contraste", "Observación": "Equivalente conceptualmente cuando la fila de hogar representa integrantes; se conserva como validación, no como cálculo principal."},
    {"Resultado": "Ingreso laboral individual positivo", "Unidad observación": "Persona", "Variable": INC_LAB, "Ponderado": "Sí", "Peso utilizado": "factor", "Clasificación": "correcto", "Observación": "Condicionado a ingreso laboral positivo."},
    {"Resultado": "Hallazgos preliminares del notebook 07", "Unidad observación": "Persona/Hogar", "Variable": "ingreso laboral y per cápita", "Ponderado": "Parcial", "Peso utilizado": "factor solo en media_pond", "Clasificación": "exploratorio", "Observación": "Medianas y cuartiles son muestrales; no mezclar con notebook 09."},
])
display(auditoria_ponderaciones)

,Resultado,Unidad observación,Variable,Ponderado,Peso utilizado,Clasificación,Observación
0,Gini nacional y regional,Hogar,ing_cor_hogar_oficial_tri,Sí,factor,correcto,Estimación puntual ponderada de hogares; bench...
1,"Media, mediana y cuantiles de ingreso corrient...",Hogar,ing_cor_hogar_oficial_tri,Sí,factor,correcto,Montos nominales trimestrales; no deflactado.
2,Ingreso corriente per cápita entre hogares,Hogar,ing_cor_pc_oficial_tri,Sí,factor,correcto si el estimando son hogares,Pregunta: cómo se distribuye el ingreso per cá...
3,Ingreso corriente per cápita entre personas,Persona,ing_cor_hogar_pc_oficial_tri,Sí,factor,estimando principal,Cada fila persona hereda el ingreso per cápita...
4,Ingreso corriente per cápita entre personas de...,Hogar,ing_cor_pc_oficial_tri,Sí,factor * tot_integ,correcto para contraste,Equivalente conceptualmente cuando la fila de ...
5,Ingreso laboral individual positivo,Persona,ingreso_persona_laboral_negocio_tri,Sí,factor,correcto,Condicionado a ingreso laboral positivo.
6,Hallazgos preliminares del notebook 07,Persona/Hogar,ingreso laboral y per cápita,Parcial,factor solo en media_pond,exploratorio,Medianas y cuartiles son muestrales; no mezcla...


## Benchmark Banxico 2018-2022

Los valores de Banxico se capturan tal como aparecen en la tabla oficial, en escala 0-100.

In [8]:
banxico_gini = pd.DataFrame([
    ("Norte", 2018, 43.1), ("Norte", 2020, 43.7), ("Norte", 2022, 40.3),
    ("Centro Norte", 2018, 43.2), ("Centro Norte", 2020, 41.3), ("Centro Norte", 2022, 40.4),
    ("Centro", 2018, 45.1), ("Centro", 2020, 44.5), ("Centro", 2022, 42.1),
    ("Sur", 2018, 47.5), ("Sur", 2020, 45.8), ("Sur", 2022, 44.9),
    ("Nacional", 2018, 45.7), ("Nacional", 2020, 45.0), ("Nacional", 2022, 43.1),
], columns=["region_banxico", "anio", "gini_banxico_0_100"])
display(banxico_gini)

,region_banxico,anio,gini_banxico_0_100
0,Norte,2018,43.1
1,Norte,2020,43.7
2,Norte,2022,40.3
3,Centro Norte,2018,43.2
4,Centro Norte,2020,41.3
5,Centro Norte,2022,40.4
6,Centro,2018,45.1
7,Centro,2020,44.5
8,Centro,2022,42.1
9,Sur,2018,47.5


## Gini nacional

Este es el cálculo principal con `ing_cor_hogar_oficial_tri`. La lectura temporal debe hacerse con cautela porque los ingresos están en pesos nominales; el Gini dentro de cada año no depende de multiplicar todos los ingresos por una constante, pero la comparación temporal sustantiva sí requiere más contexto.

In [9]:
gini_nacional = weighted_summary(hogar, ["anio"], INC_HH)
display(money_cols(gini_nacional[["anio", "n", "n_ponderado", "media", "mediana", "gini", "gini_0_100"]]))

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.lineplot(data=gini_nacional, x="anio", y="gini_0_100", marker="o", ax=ax)
ax.set_title("Gini nacional ponderado: ingreso corriente total del hogar")
ax.set_xlabel("Año")
ax.set_ylabel("Gini (0-100)")
ax.set_xticks(YEARS)
display(fig)
plt.close(fig)

,anio,n,n_ponderado,media,mediana,gini,gini_0_100
0,2018,74647,34400515.0,49851.01,35647.67,0.44,43.83
1,2020,89006,35749659.0,50309.31,36623.02,0.43,42.60
2,2022,90102,37560123.0,63695.46,47328.31,0.41,41.27
3,2024,91414,38830230.0,77863.84,59217.06,0.40,40.06


<Figure size 800x450 with 1 Axes>

## Gini por región Banxico y validación

La validación compara dirección, orden relativo y magnitud aproximada. No se fuerza el resultado a coincidir con Banxico.

In [10]:
gini_region = weighted_summary(hogar, ["anio", "region_banxico"], INC_HH)
gini_region_display = money_cols(gini_region[["anio", "region_banxico", "n", "media", "mediana", "gini_0_100"]].sort_values(["anio", "region_banxico"]))
display(gini_region_display)

propio_para_banxico = pd.concat([
    gini_region[["anio", "region_banxico", "gini_0_100"]],
    gini_nacional.assign(region_banxico="Nacional")[["anio", "region_banxico", "gini_0_100"]],
], ignore_index=True)
comparacion_banxico = propio_para_banxico.merge(banxico_gini, on=["anio", "region_banxico"], how="inner")
comparacion_banxico["dif_puntos"] = comparacion_banxico["gini_0_100"] - comparacion_banxico["gini_banxico_0_100"]
comparacion_banxico["abs_dif_puntos"] = comparacion_banxico["dif_puntos"].abs()
display(money_cols(comparacion_banxico.sort_values(["anio", "region_banxico"])))

discrepancia_max = comparacion_banxico.loc[comparacion_banxico["abs_dif_puntos"].idxmax()].copy()
display(Markdown(
    f"**Mayor discrepancia:** {discrepancia_max['region_banxico']} {int(discrepancia_max['anio'])}, "
    f"{discrepancia_max['dif_puntos']:.2f} puntos frente a Banxico."
))

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=gini_region, x="anio", y="gini_0_100", hue="region_banxico", marker="o", ax=ax)
ax.set_title("Gini por región Banxico (Gini ponderado)")
ax.set_xlabel("Año")
ax.set_ylabel("Gini (0-100)")
ax.set_xticks(YEARS)
ax.legend(title="Región")
save_doc_figure(fig, "gini_regiones_2018_2024.png", "Evolución del Gini por región Banxico")

,anio,region_banxico,n,media,mediana,gini_0_100
0,2018,Centro,18425,52910.58,37598.13,42.97
1,2018,Centro Norte,22697,52175.75,38187.22,41.98
2,2018,Norte,17334,58698.97,42651.55,42.24
3,2018,Sur,16191,35063.65,24587.57,44.86
4,2020,Centro,22067,51086.41,37232.75,41.74
5,2020,Centro Norte,26797,52504.77,39967.01,39.45
6,2020,Norte,20869,62594.35,45049.16,42.98
7,2020,Sur,19273,36612.95,26317.54,42.58
8,2022,Centro,22088,63798.94,47367.22,40.22
9,2022,Centro Norte,26790,66309.01,50991.41,39.20


,anio,region_banxico,gini_0_100,gini_banxico_0_100,dif_puntos,abs_dif_puntos
0,2018,Centro,42.97,45.1,-2.134951,2.134951
1,2018,Centro Norte,41.98,43.2,-1.215937,1.215937
12,2018,Nacional,43.83,45.7,-1.870107,1.870107
2,2018,Norte,42.24,43.1,-0.860673,0.860673
3,2018,Sur,44.86,47.5,-2.639116,2.639116
4,2020,Centro,41.74,44.5,-2.756589,2.756589
5,2020,Centro Norte,39.45,41.3,-1.853608,1.853608
13,2020,Nacional,42.60,45.0,-2.402237,2.402237
6,2020,Norte,42.98,43.7,-0.720381,0.720381
7,2020,Sur,42.58,45.8,-3.220738,3.220738


**Mayor discrepancia:** Sur 2020, -3.22 puntos frente a Banxico.

![Evolución del Gini por región Banxico](../reports/figures_documentacion/gini_regiones_2018_2024.png)

WindowsPath('C:/Users/lucia/OneDrive/Escritorio/Fer/inegi-income-modeling/reports/figures_documentacion/gini_regiones_2018_2024.png')

## Auditoría de reproducción del Gini de Banco de México

Primero se calcula el Gini directamente desde `concentradohogar.csv` con la misma variable (`ing_cor`), el mismo factor y el mismo universo. Si coincide con el mart, la discrepancia con Banxico no viene de la construcción de la base analítica.


In [11]:
def read_concentradohogar_raw(year):
    raw = pd.read_csv(RAW / str(year) / "concentradohogar.csv", usecols=["folioviv", "foliohog", "ubica_geo", "factor", "ing_cor", "tot_integ"], low_memory=False)
    raw["anio"] = year
    raw["folioviv"] = raw["folioviv"].astype(str)
    raw["foliohog"] = raw["foliohog"].astype(str)
    raw["factor"] = pd.to_numeric(raw["factor"], errors="coerce")
    raw["ing_cor"] = pd.to_numeric(raw["ing_cor"], errors="coerce")
    raw["tot_integ"] = pd.to_numeric(raw["tot_integ"], errors="coerce")
    return raw

raw_concentrado = pd.concat([read_concentradohogar_raw(year) for year in YEARS], ignore_index=True)
gini_raw = weighted_summary(raw_concentrado, ["anio"], "ing_cor").rename(columns={"gini_0_100": "gini_concentradohogar_0_100"})
mart_vs_raw = gini_nacional[["anio", "gini_0_100"]].merge(gini_raw[["anio", "gini_concentradohogar_0_100"]], on="anio")
mart_vs_raw["diferencia_pp"] = mart_vs_raw["gini_0_100"] - mart_vs_raw["gini_concentradohogar_0_100"]
display(mart_vs_raw.round(4))

checks = []
for year in YEARS:
    raw_year = raw_concentrado[raw_concentrado["anio"].eq(year)]
    mart_year = hogar[hogar["anio"].eq(year)][["folioviv", "foliohog", "factor", INC_HH]] if {"folioviv", "foliohog"}.issubset(hogar.columns) else pd.read_csv(HOGAR_PATH, usecols=["anio", "folioviv", "foliohog", "factor", INC_HH], low_memory=False).query("anio == @year")
    mart_year["folioviv"] = mart_year["folioviv"].astype(str)
    mart_year["foliohog"] = mart_year["foliohog"].astype(str)
    merged = raw_year.merge(mart_year, on=["folioviv", "foliohog"], how="left", suffixes=("_raw", "_mart"))
    checks.append({"anio": year, "filas_raw": len(raw_year), "filas_mergeadas": int(merged[INC_HH].notna().sum()), "max_abs_dif_ingreso": float((merged["ing_cor"] - merged[INC_HH]).abs().max()), "max_abs_dif_factor": float((merged["factor_raw"] - merged["factor_mart"]).abs().max())})
mart_raw_check = pd.DataFrame(checks)
display(mart_raw_check.round(4))


,anio,gini_0_100,gini_concentradohogar_0_100,diferencia_pp
0,2018,43.8299,43.8299,0.0
1,2020,42.5978,42.5978,0.0
2,2022,41.2677,41.2677,0.0
3,2024,40.0624,40.0624,0.0


,anio,filas_raw,filas_mergeadas,max_abs_dif_ingreso,max_abs_dif_factor
0,2018,74647,74647,0.0,0.0
1,2020,89006,89006,0.0,0.0
2,2022,90102,90102,0.0,0.0
3,2024,91414,91414,0.0,0.0


## Ponderado vs no ponderado y diagnóstico de definición

El cambio de ponderar no explica por sí solo la distancia frente a Banxico. La comparación principal se mantiene en hogares: `ing_cor_hogar_oficial_tri` + `factor`. Además se conserva una variante diagnóstica de ingreso corriente per cápita distribuido **entre personas**, calculada ahora desde `mart_persona` con `factor`. Esta variante ayuda a diagnosticar definiciones, pero no sustituye la definición principal hasta reproducir exactamente las bases CONEVAL usadas por Banxico.

In [12]:
def weighted_summary_custom(df, group_cols, income_col, weight_col, positive_only=False):
    work = df[df[income_col].gt(0)].copy() if positive_only else df.copy()
    rows = []
    for key, group in work.groupby(group_cols, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        row = dict(zip(group_cols, key))
        x = pd.to_numeric(group[income_col], errors="coerce")
        w = pd.to_numeric(group[weight_col], errors="coerce")
        mask = x.notna() & w.notna() & w.gt(0) & x.ge(0)
        x, w = x[mask], w[mask]
        qs = weighted_quantile(x, w, [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
        row.update({"n_muestral": int(mask.sum()), "poblacion_expandida": float(w.sum()), "media_ponderada": float(np.average(x, weights=w)) if len(x) else np.nan, "p10": qs[0], "p25": qs[1], "mediana_ponderada": qs[2], "p75": qs[3], "p90": qs[4], "p95": qs[5], "p99": qs[6], "maximo": float(np.nanmax(x)) if len(x) else np.nan, "gini_ponderado_0_100": weighted_gini(x, w) * 100})
        rows.append(row)
    return pd.DataFrame(rows)

weighted_unweighted = []
for year, group in hogar[hogar["anio"].isin([2018, 2020, 2022])].groupby("anio"):
    x = pd.to_numeric(group[INC_HH], errors="coerce")
    w_unweighted = pd.Series(np.ones(len(group)), index=group.index)
    weighted_unweighted.append({"anio": year, "gini_ponderado_0_100": weighted_gini(x, group["factor"]) * 100, "gini_no_ponderado_0_100": weighted_gini(x, w_unweighted) * 100})
weighted_unweighted = pd.DataFrame(weighted_unweighted).merge(banxico_gini[banxico_gini["region_banxico"].eq("Nacional")][["anio", "gini_banxico_0_100"]], on="anio")
display(weighted_unweighted.round(2))

hogar_gini_rows = []
for year, group in hogar.groupby("anio"):
    hogar_gini_rows.append({"region_banxico": "Nacional", "anio": year, "gini_hogar_total_factor": weighted_gini(group[INC_HH], group["factor"]) * 100})
for (year, region), group in hogar.groupby(["anio", "region_banxico"]):
    hogar_gini_rows.append({"region_banxico": region, "anio": year, "gini_hogar_total_factor": weighted_gini(group[INC_HH], group["factor"]) * 100})

pc_persona_rows = []
for year, group in persona.groupby("anio"):
    pc_persona_rows.append({"region_banxico": "Nacional", "anio": year, "gini_pc_persona_factor": weighted_gini(group[INC_PC_PERSONA], group["factor"]) * 100})
for (year, region), group in persona.groupby(["anio", "region_banxico"]):
    pc_persona_rows.append({"region_banxico": region, "anio": year, "gini_pc_persona_factor": weighted_gini(group[INC_PC_PERSONA], group["factor"]) * 100})

diag_banxico = pd.DataFrame(hogar_gini_rows).merge(pd.DataFrame(pc_persona_rows), on=["region_banxico", "anio"], how="left").merge(banxico_gini, on=["region_banxico", "anio"], how="inner")
diag_banxico["dif_hogar_total_pp"] = diag_banxico["gini_hogar_total_factor"] - diag_banxico["gini_banxico_0_100"]
diag_banxico["dif_pc_persona_pp"] = diag_banxico["gini_pc_persona_factor"] - diag_banxico["gini_banxico_0_100"]
display(diag_banxico.sort_values(["anio", "region_banxico"]).round(2))

fig, ax = plt.subplots(figsize=(9, 5))
nacional_diag = diag_banxico[diag_banxico["region_banxico"].eq("Nacional")].sort_values("anio")
ax.plot(nacional_diag["anio"], nacional_diag["gini_banxico_0_100"], "s--", linewidth=2.2, label="Banxico")
ax.plot(nacional_diag["anio"], nacional_diag["gini_hogar_total_factor"], "o-", linewidth=2.2, label="Mart: hogar total + factor")
ax.plot(nacional_diag["anio"], nacional_diag["gini_pc_persona_factor"], "^-", linewidth=2.2, label="Diagnóstico: per cápita persona + factor")
ax.set_title("Gini nacional propio vs Banxico")
ax.set_xlabel("Año")
ax.set_ylabel("Gini (0-100)")
ax.set_xticks([2018, 2020, 2022])
ax.legend()
ax.text(0.01, -0.20, "La variante per cápita es diagnóstica; no reemplaza la definición primaria sin validar CONEVAL.", transform=ax.transAxes, fontsize=9)
save_doc_figure(fig, "gini_banxico_comparacion_2018_2022.png", "Gini nacional propio vs Banxico")

,anio,gini_ponderado_0_100,gini_no_ponderado_0_100,gini_banxico_0_100
0,2018,43.83,42.73,45.7
1,2020,42.60,42.05,45.0
2,2022,41.27,40.95,43.1


,region_banxico,anio,gini_hogar_total_factor,gini_pc_persona_factor,gini_banxico_0_100,dif_hogar_total_pp,dif_pc_persona_pp
3,Centro,2018,42.97,45.92,45.1,-2.13,0.82
4,Centro Norte,2018,41.98,43.83,43.2,-1.22,0.63
0,Nacional,2018,43.83,46.29,45.7,-1.87,0.59
5,Norte,2018,42.24,43.97,43.1,-0.86,0.87
6,Sur,2018,44.86,47.33,47.5,-2.64,-0.17
7,Centro,2020,41.74,44.54,44.5,-2.76,0.04
8,Centro Norte,2020,39.45,41.54,41.3,-1.85,0.24
1,Nacional,2020,42.60,45.05,45.0,-2.40,0.05
9,Norte,2020,42.98,44.21,43.7,-0.72,0.51
10,Sur,2020,42.58,45.52,45.8,-3.22,-0.28


![Gini nacional propio vs Banxico](../reports/figures_documentacion/gini_banxico_comparacion_2018_2022.png)

WindowsPath('C:/Users/lucia/OneDrive/Escritorio/Fer/inegi-income-modeling/reports/figures_documentacion/gini_banxico_comparacion_2018_2022.png')

### Reglas definitivas de ponderación

La tabla siguiente fija el principio que queda para el resto del proyecto: antes de calcular una media, mediana, cuantil, Gini, proporción o brecha se debe declarar unidad de observación, población objetivo, variable de peso y definición del estimando.

In [13]:
tabla_definitiva_ponderacion = pd.DataFrame([
    {"Estimando": "Media ingreso hogar", "Unidad de la tabla": "Hogar", "Peso": "factor", "Interpretación": "Hogares"},
    {"Estimando": "Mediana ingreso hogar", "Unidad de la tabla": "Hogar", "Peso": "factor", "Interpretación": "Hogares"},
    {"Estimando": "Media ingreso individual", "Unidad de la tabla": "Persona", "Peso": "factor", "Interpretación": "Personas"},
    {"Estimando": "Mediana ingreso individual", "Unidad de la tabla": "Persona", "Peso": "factor", "Interpretación": "Personas"},
    {"Estimando": "Población total desde hogares", "Unidad de la tabla": "Hogar", "Peso": "factor × tot_integ", "Interpretación": "Personas integrantes del hogar"},
    {"Estimando": "Ingreso PC hogar distribuido entre hogares", "Unidad de la tabla": "Hogar", "Peso": "factor", "Interpretación": "Hogares"},
    {"Estimando": "Ingreso PC hogar distribuido entre personas", "Unidad de la tabla": "Hogar", "Peso": "factor × tot_integ", "Interpretación": "Personas integrantes del hogar"},
    {"Estimando": "Ingreso PC hogar distribuido entre personas desde mart_persona", "Unidad de la tabla": "Persona", "Peso": "factor", "Interpretación": "Personas"},
])
display(tabla_definitiva_ponderacion)

persona_con_tot = persona[["anio", "folioviv", "foliohog", WEIGHT, INC_PC_PERSONA]].merge(
    hogar[["anio", "folioviv", "foliohog", "tot_integ", INC_PC, W_PERSONAS_DESDE_HOGAR]],
    on=["anio", "folioviv", "foliohog"], how="left"
)
persona_con_tot["factor_x_tot_integ_sobre_personas"] = persona_con_tot[WEIGHT] * persona_con_tot["tot_integ"]
auditoria_doble_ponderacion = persona_con_tot.groupby("anio").agg(
    suma_factor_sobre_filas_persona=(WEIGHT, "sum"),
    suma_factor_x_tot_integ_sobre_filas_persona=("factor_x_tot_integ_sobre_personas", "sum"),
).reset_index()
auditoria_doble_ponderacion["razon_error"] = auditoria_doble_ponderacion["suma_factor_x_tot_integ_sobre_filas_persona"] / auditoria_doble_ponderacion["suma_factor_sobre_filas_persona"]
display(auditoria_doble_ponderacion.round(2))

pc_hogar_personas = weighted_summary_custom(hogar, ["anio"], INC_PC, W_PERSONAS_DESDE_HOGAR).rename(columns={"poblacion_expandida": "poblacion_hogar_x_integrantes", "mediana_ponderada": "mediana_hogar_x_integrantes", "gini_ponderado_0_100": "gini_hogar_x_integrantes"})
pc_persona_directo = weighted_summary_custom(persona, ["anio"], INC_PC_PERSONA, WEIGHT).rename(columns={"poblacion_expandida": "poblacion_persona_factor", "mediana_ponderada": "mediana_persona_factor", "gini_ponderado_0_100": "gini_persona_factor"})
equivalencia_pc = pc_hogar_personas[["anio", "n_muestral", "poblacion_hogar_x_integrantes", "mediana_hogar_x_integrantes", "gini_hogar_x_integrantes"]].merge(
    pc_persona_directo[["anio", "n_muestral", "poblacion_persona_factor", "mediana_persona_factor", "gini_persona_factor"]],
    on="anio", suffixes=("_hogares", "_personas")
)
equivalencia_pc["dif_poblacion"] = equivalencia_pc["poblacion_persona_factor"] - equivalencia_pc["poblacion_hogar_x_integrantes"]
equivalencia_pc["dif_mediana"] = equivalencia_pc["mediana_persona_factor"] - equivalencia_pc["mediana_hogar_x_integrantes"]
equivalencia_pc["dif_gini"] = equivalencia_pc["gini_persona_factor"] - equivalencia_pc["gini_hogar_x_integrantes"]
display(equivalencia_pc.round(4))

auditoria_usos_tot_integ = pd.DataFrame([
    {"artefacto": "notebook 07", "uso factor × tot_integ": "no utilizado", "clasificación": "no utilizado", "nota": "Las medianas/cuartiles son muestrales; media_pond usa factor."},
    {"artefacto": "notebook 09", "uso factor × tot_integ": "hogar -> personas", "clasificación": "correcto como contraste", "nota": "Se conserva para explicar el estimando desde hogares, no se usa sobre mart_persona."},
    {"artefacto": "notebook 09", "uso factor × tot_integ": "sobre mart_persona", "clasificación": "incorrecto si apareciera", "nota": "Auditoría confirma que no se usa para resultados finales."},
    {"artefacto": "documentación final", "uso factor × tot_integ": "texto previo ambiguo", "clasificación": "requiere corrección", "nota": "Se reemplaza por la regla explícita: personas = mart_persona + factor."},
])
display(auditoria_usos_tot_integ)

,Estimando,Unidad de la tabla,Peso,Interpretación
0,Media ingreso hogar,Hogar,factor,Hogares
1,Mediana ingreso hogar,Hogar,factor,Hogares
2,Media ingreso individual,Persona,factor,Personas
3,Mediana ingreso individual,Persona,factor,Personas
4,Población total desde hogares,Hogar,factor × tot_integ,Personas integrantes del hogar
5,Ingreso PC hogar distribuido entre hogares,Hogar,factor,Hogares
6,Ingreso PC hogar distribuido entre personas,Hogar,factor × tot_integ,Personas integrantes del hogar
7,Ingreso PC hogar distribuido entre personas de...,Persona,factor,Personas


,anio,suma_factor_sobre_filas_persona,suma_factor_x_tot_integ_sobre_filas_persona,razon_error
0,2018,123934029,561257240,4.53
1,2020,126838467,566104426,4.46
2,2022,128999038,558856484,4.33
3,2024,130325969,554726112,4.26


,anio,n_muestral_hogares,poblacion_hogar_x_integrantes,mediana_hogar_x_integrantes,gini_hogar_x_integrantes,n_muestral_personas,poblacion_persona_factor,mediana_persona_factor,gini_persona_factor,dif_poblacion,dif_mediana,dif_gini
0,2018,74647,123836081.0,9294.4395,46.0075,269206,123934029.0,9297.6009,46.2860,97948.0,3.1614,0.2785
1,2020,89006,126760856.0,9710.8845,44.9537,315743,126838467.0,9717.9100,45.0485,77611.0,7.0255,0.0948
2,2022,90102,128889708.0,12979.3370,43.6998,309684,128999038.0,12989.1782,44.0356,109330.0,9.8412,0.3358
3,2024,91414,130226218.0,16596.6406,42.8724,308598,130325969.0,16602.0210,43.0490,99751.0,5.3805,0.1766


,artefacto,uso factor × tot_integ,clasificación,nota
0,notebook 07,no utilizado,no utilizado,Las medianas/cuartiles son muestrales; media_p...
1,notebook 09,hogar -> personas,correcto como contraste,Se conserva para explicar el estimando desde h...
2,notebook 09,sobre mart_persona,incorrecto si apareciera,Auditoría confirma que no se usa para resultad...
3,documentación final,texto previo ambiguo,requiere corrección,Se reemplaza por la regla explícita: personas ...


## Cola de la distribución

Se revisan percentiles ponderados y máximos por año. No se eliminan outliers ni se winsoriza; la meta es diagnosticar si la diferencia con Banxico parece venir de colas, universo o definición de ingreso.


In [14]:
colas_ingreso = weighted_summary_custom(hogar, ["anio"], INC_HH, "factor")[["anio", "p10", "p25", "mediana_ponderada", "p75", "p90", "p95", "p99", "maximo"]]
display(colas_ingreso.round(2))


,anio,p10,p25,mediana_ponderada,p75,p90,p95,p99,maximo
0,2018,13210.50,21490.30,35647.67,58849.57,95066.45,131235.39,244437.25,4501830.28
1,2020,13955.46,22304.34,36623.02,60361.83,96105.67,130671.44,239166.75,10702107.40
2,2022,18735.25,29200.76,47328.31,76356.38,119979.18,160048.03,296348.17,7153770.46
3,2024,23636.75,36874.38,59217.06,95142.00,145807.42,190156.66,343348.95,17431977.54


## Tabla final de validación Banxico

La conclusión se clasifica como reproducción cercana cuando la diferencia absoluta es menor o igual a 1 punto de Gini; en caso contrario se conserva como reproducción parcial.


In [15]:
comparacion_banxico_final = diag_banxico[["anio", "region_banxico", "gini_hogar_total_factor", "gini_banxico_0_100", "dif_hogar_total_pp"]].copy()
comparacion_banxico_final = comparacion_banxico_final.rename(columns={"gini_hogar_total_factor": "nuestro_gini_0_100", "gini_banxico_0_100": "banxico_0_100", "dif_hogar_total_pp": "diferencia_absoluta_pp"})
comparacion_banxico_final["diferencia_relativa_pct"] = 100 * comparacion_banxico_final["diferencia_absoluta_pp"] / comparacion_banxico_final["banxico_0_100"]
comparacion_banxico_final["conclusion"] = np.where(comparacion_banxico_final["diferencia_absoluta_pp"].abs().le(1), "reproducción cercana", "reproducción parcial")
display(comparacion_banxico_final.sort_values(["anio", "region_banxico"]).round(2))


,anio,region_banxico,nuestro_gini_0_100,banxico_0_100,diferencia_absoluta_pp,diferencia_relativa_pct,conclusion
3,2018,Centro,42.97,45.1,-2.13,-4.73,reproducción parcial
4,2018,Centro Norte,41.98,43.2,-1.22,-2.81,reproducción parcial
0,2018,Nacional,43.83,45.7,-1.87,-4.09,reproducción parcial
5,2018,Norte,42.24,43.1,-0.86,-2.00,reproducción cercana
6,2018,Sur,44.86,47.5,-2.64,-5.56,reproducción parcial
7,2020,Centro,41.74,44.5,-2.76,-6.19,reproducción parcial
8,2020,Centro Norte,39.45,41.3,-1.85,-4.49,reproducción parcial
1,2020,Nacional,42.60,45.0,-2.40,-5.34,reproducción parcial
9,2020,Norte,42.98,43.7,-0.72,-1.65,reproducción cercana
10,2020,Sur,42.58,45.8,-3.22,-7.03,reproducción parcial


## Gini por entidad, tamaño de localidad y estrato socioeconómico

Estas tablas miden desigualdad dentro de cada territorio o estrato. No son brechas entre territorios; por ejemplo, un Gini alto en una entidad indica mayor dispersión interna de ingresos de hogares dentro de esa entidad.

In [16]:
gini_entidad    = weighted_summary(hogar, ["anio", "entidad"], INC_HH)
gini_tam_loc    = weighted_summary(hogar, ["anio", "tam_loc_desc"], INC_HH)
gini_est_socio  = weighted_summary(hogar, ["anio", "est_socio_desc"], INC_HH)

entidad_2024 = gini_entidad[gini_entidad["anio"].eq(2024)].sort_values("gini_0_100")
display(Markdown("### Entidades con menor Gini interno en 2024"))
display(money_cols(entidad_2024.head(8)[["entidad", "n", "mediana", "gini_0_100"]]))
display(Markdown("### Entidades con mayor Gini interno en 2024"))
display(money_cols(entidad_2024.tail(8)[["entidad", "n", "mediana", "gini_0_100"]]))

display(Markdown("### Tamaño de localidad, 2024"))
display(money_cols(gini_tam_loc[gini_tam_loc["anio"].eq(2024)][["tam_loc_desc", "n", "media", "mediana", "gini_0_100"]].sort_values("mediana", ascending=False)))

display(Markdown("### Estrato socioeconómico, 2024"))
display(money_cols(gini_est_socio[gini_est_socio["anio"].eq(2024)][["est_socio_desc", "n", "media", "mediana", "gini_0_100"]].sort_values("mediana", ascending=False)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=gini_tam_loc, x="anio", y="mediana", hue="tam_loc_desc", marker="o", ax=axes[0])
axes[0].set_title("Mediana ponderada por tamaño de localidad")
axes[0].set_xlabel("Año")
axes[0].set_ylabel("Ingreso corriente hogar")
axes[0].legend(title="Tamaño", fontsize=8)
sns.lineplot(data=gini_est_socio, x="anio", y="mediana", hue="est_socio_desc", marker="o", ax=axes[1])
axes[1].set_title("Mediana ponderada por estrato socioeconómico")
axes[1].set_xlabel("Año")
axes[1].set_ylabel("Ingreso corriente hogar")
axes[1].legend(title="Estrato", fontsize=8)
fig.tight_layout()
display(fig)
plt.close(fig)

### Entidades con menor Gini interno en 2024

,entidad,n,mediana,gini_0_100
97,Baja California,4048,82352.53,33.82
98,Baja California Sur,2803,87724.56,34.43
120,Sinaloa,3396,68831.50,34.54
118,Quintana Roo,2418,72474.34,34.91
103,Coahuila de Zaragoza,4115,71413.42,35.07
124,Tlaxcala,2255,46566.55,35.38
110,Mexico,3573,58305.93,35.56
123,Tamaulipas,2336,62538.58,35.85


### Entidades con mayor Gini interno en 2024

,entidad,n,mediana,gini_0_100
111,Michoacan de Ocampo,2130,52567.56,39.07
126,Yucatan,2805,62110.25,39.18
122,Tabasco,2301,48724.27,40.04
102,Ciudad de Mexico,2576,81865.70,40.40
107,Guerrero,2491,35973.51,40.52
119,San Luis Potosi,2695,56312.30,40.67
115,Oaxaca,2644,39191.04,41.10
114,Nuevo Leon,3796,83053.87,43.56


### Tamaño de localidad, 2024

,tam_loc_desc,n,media,mediana,gini_0_100
12,Localidades con 100 000 y más habitantes,34307,95716.30,74003.85,37.96
13,Localidades con 15 000 a 99 999 habitantes,10453,73317.53,58834.20,36.38
14,Localidades con 2 500 a 14 999 habitantes,10829,63845.17,50817.00,36.92
15,Localidades con menos de 2 500 habitantes,35825,48004.00,37009.99,38.85


### Estrato socioeconómico, 2024

,est_socio_desc,n,media,mediana,gini_0_100
12,Alto,8180,146413.17,113647.44,38.42
14,Medio alto,17916,85687.27,70257.44,33.96
15,Medio bajo,46384,64769.17,53492.47,34.34
13,Bajo,18934,44020.88,34630.41,37.67


<Figure size 1400x500 with 2 Axes>

## CDMX

Para CDMX se calculan métricas descriptivas ponderadas por año. Los ingresos están en pesos nominales trimestrales. Se distinguen dos estimandos per cápita: distribución entre hogares (`mart_hogar` + `factor`) y distribución entre personas (`mart_persona` + `factor`). Para ingreso laboral individual se usa el universo con ingreso laboral positivo; la probabilidad de tener ingreso laboral positivo debe analizarse aparte.

In [17]:
def cve_ent_2(series):
    return series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(2)

cdmx_hogar = hogar[cve_ent_2(hogar["cve_ent"]).eq("09")].copy()
cdmx_persona = persona[cve_ent_2(persona["cve_ent"]).eq("09")].copy()

cdmx_ingreso_hogar = weighted_summary(cdmx_hogar, ["anio"], INC_HH).assign(metrica="Ingreso corriente hogar")
cdmx_ingreso_pc_hogares = weighted_summary(cdmx_hogar, ["anio"], INC_PC).assign(metrica="Ingreso corriente per cápita entre hogares")
cdmx_ingreso_pc_personas = weighted_summary(cdmx_persona, ["anio"], INC_PC_PERSONA).assign(metrica="Ingreso corriente per cápita entre personas")
cdmx_laboral = weighted_summary(cdmx_persona, ["anio"], INC_LAB, positive_only=True).assign(metrica="Ingreso laboral individual positivo")
cdmx_metricas = pd.concat([cdmx_ingreso_hogar, cdmx_ingreso_pc_hogares, cdmx_ingreso_pc_personas, cdmx_laboral], ignore_index=True)

display(money_cols(cdmx_metricas[["anio", "metrica", "n", "n_ponderado", "media", "p25", "mediana", "p75", "p90", "p95", "gini_0_100"]]))

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=cdmx_metricas, x="anio", y="mediana", hue="metrica", marker="o", ax=ax)
ax.set_title("CDMX: medianas ponderadas por métrica")
ax.set_xlabel("Año")
ax.set_ylabel("Pesos nominales trimestrales")
ax.set_xticks(YEARS)
ax.legend(title="Métrica", fontsize=8)
display(fig)
plt.close(fig)

,anio,metrica,n,n_ponderado,media,p25,mediana,p75,p90,p95,gini_0_100
0,2018,Ingreso corriente hogar,2169,2778842.0,79085.34,31568.69,50578.32,86605.34,151895.29,227798.73,47.12
1,2020,Ingreso corriente hogar,2570,2731683.0,67356.70,30168.83,50901.62,83477.48,131101.10,169942.81,40.32
2,2022,Ingreso corriente hogar,2585,2990030.0,89310.27,40854.55,62747.60,101639.94,168812.28,229365.68,41.94
3,2024,Ingreso corriente hogar,2576,3082330.0,110684.85,51650.29,81865.70,134758.07,208928.66,284867.81,40.40
4,2018,Ingreso corriente per cápita entre hogares,2169,2778842.0,31719.17,10030.80,17382.34,31475.40,65390.29,104676.54,54.28
5,2020,Ingreso corriente per cápita entre hogares,2570,2731683.0,24927.05,10006.32,16565.21,29326.36,51436.44,68589.10,45.80
6,2022,Ingreso corriente per cápita entre hogares,2585,2990030.0,36847.05,14049.36,22739.21,39899.63,75254.94,106777.21,48.58
7,2024,Ingreso corriente per cápita entre hogares,2576,3082330.0,48113.98,17747.47,30292.83,54483.78,102377.79,146336.95,47.93
8,2018,Ingreso corriente per cápita entre personas,7475,9219295.0,24247.95,8494.78,13457.47,24130.95,45923.58,77798.95,51.84
9,2020,Ingreso corriente per cápita entre personas,9056,9260329.0,20032.87,8587.65,13841.17,23151.81,40109.40,53387.49,43.98


<Figure size 900x500 with 1 Axes>

## Brechas territoriales

Aquí se separa la idea de desigualdad interna y brecha territorial:

- desigualdad interna: Gini dentro de cada región, tamaño de localidad o estrato;
- brecha territorial: diferencia o razón entre medianas de grupos territoriales dentro del mismo año.

Esta no es una descomposición formal del Gini; es una lectura descriptiva para orientar la tesina.

In [18]:
def median_gap(summary_df, dim):
    rows = []
    for year, g in summary_df.groupby("anio"):
        hi = g.loc[g["mediana"].idxmax()]
        lo = g.loc[g["mediana"].idxmin()]
        rows.append({
            "anio": int(year),
            "dimension": dim,
            "grupo_mayor_mediana": hi[dim],
            "mediana_mayor": hi["mediana"],
            "grupo_menor_mediana": lo[dim],
            "mediana_menor": lo["mediana"],
            "razon_mediana": hi["mediana"] / lo["mediana"],
            "brecha_mediana": hi["mediana"] - lo["mediana"],
        })
    return pd.DataFrame(rows)

brechas = pd.concat([
    median_gap(gini_region, "region_banxico"),
    median_gap(gini_tam_loc, "tam_loc_desc"),
    median_gap(gini_est_socio, "est_socio_desc"),
], ignore_index=True)
display(money_cols(brechas.sort_values(["dimension", "anio"])))

fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=brechas, x="anio", y="razon_mediana", hue="dimension", marker="o", ax=ax)
ax.set_title("Brecha territorial: razón entre mayor y menor mediana")
ax.set_xlabel("Año")
ax.set_ylabel("Razón de medianas")
ax.set_xticks(YEARS)
ax.legend(title="Dimensión")
display(fig)
plt.close(fig)

,anio,dimension,grupo_mayor_mediana,mediana_mayor,grupo_menor_mediana,mediana_menor,razon_mediana,brecha_mediana
8,2018,est_socio_desc,Alto,74047.434816,Bajo,19537.120000,3.790090,54510.314816
9,2020,est_socio_desc,Alto,70690.170696,Bajo,21891.868864,3.229061,48798.301833
10,2022,est_socio_desc,Alto,90954.465452,Bajo,28510.432208,3.190217,62444.033245
11,2024,est_socio_desc,Alto,113647.436773,Bajo,34630.411333,3.281724,79017.025439
0,2018,region_banxico,Norte,42651.545857,Sur,24587.567399,1.734679,18063.978458
1,2020,region_banxico,Norte,45049.160000,Sur,26317.541803,1.711754,18731.618197
2,2022,region_banxico,Norte,60807.726020,Sur,34302.872073,1.772672,26504.853947
3,2024,region_banxico,Norte,75412.299226,Sur,42532.580035,1.773048,32879.719191
4,2018,tam_loc_desc,Localidades con 100 000 y más habitantes,45223.037276,Localidades con menos de 2 500 habitantes,22330.298852,2.025187,22892.738424
5,2020,tam_loc_desc,Localidades con 100 000 y más habitantes,44782.531998,Localidades con menos de 2 500 habitantes,24621.121489,1.818866,20161.410509


<Figure size 900x500 with 1 Axes>

## Delimitación oficial de zonas metropolitanas

No se comparan municipios sueltos. Se usa `docs/zonas_metropolitanas_prioritarias_2020.csv`, un extracto versionable de **Las metrópolis de México 2020** de CONAPO/SEDATU/INEGI. La fuente etiqueta la zona como “Ciudad de México”; aquí se reporta como “Valle de México” para mantener el término de trabajo, conservando `nombre_oficial` en el mapping.


In [19]:
zonas_prioritarias = ["Valle de México", "Guadalajara", "Monterrey"]
mapa_zm = pd.read_csv(ZM_MAP_PATH, dtype={"cve_ent": "string", "cve_mun": "string", "clave_compuesta_municipio": "string"})
mapa_zm["cve_ent"] = mapa_zm["cve_ent"].str.zfill(2)
mapa_zm["cve_mun"] = mapa_zm["cve_mun"].str.zfill(3)

resumen_mapa_zm = mapa_zm.groupby(["zona_metropolitana", "nombre_oficial", "tipo", "anio_delimitacion"]).agg(
    municipios_oficiales=("clave_compuesta_municipio", "nunique"),
    entidades=("entidad", lambda s: ", ".join(sorted(s.dropna().unique()))),
).reset_index()
display(resumen_mapa_zm)

display(mapa_zm[["zona_metropolitana", "cve_ent", "entidad", "cve_mun", "municipio"]].head(12))


,zona_metropolitana,nombre_oficial,tipo,anio_delimitacion,municipios_oficiales,entidades
0,Guadalajara,Guadalajara,Zona metropolitana,2020,7,Jalisco
1,Monterrey,Monterrey,Zona metropolitana,2020,16,Nuevo León
2,Valle de México,Ciudad de México,Zona metropolitana,2020,63,"Ciudad de México, Hidalgo, México"


,zona_metropolitana,cve_ent,entidad,cve_mun,municipio
0,Guadalajara,14,Jalisco,039,Guadalajara
1,Guadalajara,14,Jalisco,051,Juanacatlán
2,Guadalajara,14,Jalisco,070,El Salto
3,Guadalajara,14,Jalisco,097,Tlajomulco de Zúñiga
4,Guadalajara,14,Jalisco,098,San Pedro Tlaquepaque
5,Guadalajara,14,Jalisco,101,Tonalá
6,Guadalajara,14,Jalisco,120,Zapopan
7,Monterrey,19,Nuevo León,006,Apodaca
8,Monterrey,19,Nuevo León,009,Cadereyta Jiménez
9,Monterrey,19,Nuevo León,010,El Carmen


## Cobertura metropolitana en ENIGH

Se valida cobertura por zona y año antes de interpretar: hogares, personas, población expandida, municipios presentes y combinaciones `est_dis + upm`. La lectura es descriptiva exploratoria, no inferencia formal con diseño muestral complejo.


In [20]:
hogar_geo_cols = ["anio", "folioviv", "foliohog", "cve_ent", "cve_mun", "entidad", "municipio", "region_banxico", "tam_loc_desc", "est_socio_desc", "est_dis", "upm", "factor", "tot_integ", INC_HH, INC_PC]
persona_geo_cols = ["anio", "folioviv", "foliohog", "numren", "cve_ent", "cve_mun", "entidad", "municipio", "region_banxico", "tam_loc_desc", "est_socio_desc", "factor", INC_PC_PERSONA, INC_LAB]
hogar_geo = pd.read_csv(HOGAR_PATH, usecols=hogar_geo_cols, low_memory=False)
persona_geo = pd.read_csv(PERSONA_PATH, usecols=persona_geo_cols, low_memory=False)

for df in [hogar_geo, persona_geo]:
    df["cve_ent"] = as_code(df["cve_ent"], 2)
    df["cve_mun"] = as_code(df["cve_mun"], 3)
for col in ["anio", "factor", "tot_integ", INC_HH, INC_PC]:
    if col in hogar_geo.columns:
        hogar_geo[col] = pd.to_numeric(hogar_geo[col], errors="coerce")
for col in ["anio", "factor", INC_PC_PERSONA, INC_LAB]:
    persona_geo[col] = pd.to_numeric(persona_geo[col], errors="coerce")
hogar_geo[W_PERSONAS_DESDE_HOGAR] = hogar_geo["factor"] * hogar_geo["tot_integ"]

mapa_keys = mapa_zm[["cve_ent", "cve_mun", "zona_metropolitana"]].drop_duplicates()
hogar_geo = hogar_geo.merge(mapa_keys, on=["cve_ent", "cve_mun"], how="left")
persona_geo = persona_geo.merge(mapa_keys, on=["cve_ent", "cve_mun"], how="left")
hogar_zm = hogar_geo[hogar_geo["zona_metropolitana"].isin(zonas_prioritarias)].copy()
persona_zm = persona_geo[persona_geo["zona_metropolitana"].isin(zonas_prioritarias)].copy()

oficiales = mapa_zm.groupby("zona_metropolitana")["clave_compuesta_municipio"].nunique().to_dict()
coverage_rows = []
for (zona, year), group in hogar_zm.groupby(["zona_metropolitana", "anio"]):
    personas_group = persona_zm[persona_zm["zona_metropolitana"].eq(zona) & persona_zm["anio"].eq(year)]
    coverage_rows.append({"zona_metropolitana": zona, "anio": int(year), "n_hogares": len(group), "n_personas": len(personas_group), "poblacion_expandida": float(personas_group["factor"].sum()), "municipios_oficiales": int(oficiales[zona]), "municipios_presentes_enigh": group[["cve_ent", "cve_mun"]].drop_duplicates().shape[0], "upm_diseno": group[["est_dis", "upm"]].drop_duplicates().shape[0]})
cobertura_zm = pd.DataFrame(coverage_rows).sort_values(["zona_metropolitana", "anio"])
display(cobertura_zm.round(0))

,zona_metropolitana,anio,n_hogares,n_personas,poblacion_expandida,municipios_oficiales,municipios_presentes_enigh,upm_diseno
0,Guadalajara,2018,1011,3666,4524082.0,7,6,204
1,Guadalajara,2020,1488,5220,5215551.0,7,6,252
2,Guadalajara,2022,1342,4666,5044533.0,7,5,248
3,Guadalajara,2024,1373,4487,6021378.0,7,7,277
4,Monterrey,2018,2157,7566,5095822.0,16,13,355
5,Monterrey,2020,2558,8913,5461146.0,16,14,429
6,Monterrey,2022,2572,8541,5465267.0,16,16,427
7,Monterrey,2024,2509,8264,5743578.0,16,15,419
8,Valle de México,2018,3736,13128,20520047.0,63,48,603
9,Valle de México,2020,4701,16667,22113791.0,63,52,732


## CDMX: entidad federativa vs Zona Metropolitana del Valle de México

CDMX es una entidad federativa. La Zona Metropolitana del Valle de México incluye CDMX y municipios metropolitanos de México e Hidalgo en la delimitación 2020 usada aquí. No son unidades equivalentes.


In [21]:
def add_metric_custom(df, label, value_col, weight_col, positive_only=False):
    out = weighted_summary_custom(df, ["anio"], value_col, weight_col, positive_only=positive_only)
    out["metrica"] = label
    return out

cdmx_hogar_geo = hogar_geo[hogar_geo["cve_ent"].eq("09")].copy()
cdmx_persona_geo = persona_geo[persona_geo["cve_ent"].eq("09")].copy()
zmvm_persona = persona_geo[persona_geo["zona_metropolitana"].eq("Valle de México")].copy()

cdmx_metricas_ext = pd.concat([
    add_metric_custom(cdmx_hogar_geo, "CDMX entidad - ingreso corriente hogar", INC_HH, "factor"),
    add_metric_custom(cdmx_persona_geo, "CDMX entidad - ingreso corriente per cápita entre personas", INC_PC_PERSONA, "factor"),
    add_metric_custom(zmvm_persona, "ZM Valle de México - ingreso corriente per cápita entre personas", INC_PC_PERSONA, "factor"),
    add_metric_custom(cdmx_persona_geo, "CDMX entidad - ingreso laboral individual positivo", INC_LAB, "factor", positive_only=True),
], ignore_index=True)
display(cdmx_metricas_ext[["anio", "metrica", "n_muestral", "poblacion_expandida", "media_ponderada", "p25", "mediana_ponderada", "p75", "p90", "p95", "gini_ponderado_0_100"]].round(2))

cdmx_plot = cdmx_metricas_ext[cdmx_metricas_ext["anio"].eq(2024) & cdmx_metricas_ext["metrica"].isin(["CDMX entidad - ingreso corriente per cápita entre personas", "ZM Valle de México - ingreso corriente per cápita entre personas"])].copy()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(cdmx_plot["metrica"], cdmx_plot["mediana_ponderada"], color=["#457b9d", "#e29578"])
ax.set_title("CDMX entidad vs ZM Valle de México, 2024")
ax.set_xlabel("Mediana ponderada, pesos nominales trimestrales")
ax.set_ylabel("Unidad territorial")
ax.text(0.01, -0.22, "Ingreso corriente per cápita del hogar distribuido entre personas; filas persona + factor.", transform=ax.transAxes, fontsize=9)
save_doc_figure(fig, "cdmx_vs_zmvm_2024.png", "CDMX entidad vs Zona Metropolitana del Valle de México, 2024")

,anio,metrica,n_muestral,poblacion_expandida,media_ponderada,p25,mediana_ponderada,p75,p90,p95,gini_ponderado_0_100
0,2018,CDMX entidad - ingreso corriente hogar,2169,2778842.0,79085.34,31568.69,50578.32,86605.34,151895.29,227798.73,47.12
1,2020,CDMX entidad - ingreso corriente hogar,2570,2731683.0,67356.70,30168.83,50901.62,83477.48,131101.10,169942.81,40.32
2,2022,CDMX entidad - ingreso corriente hogar,2585,2990030.0,89310.27,40854.55,62747.60,101639.94,168812.28,229365.68,41.94
3,2024,CDMX entidad - ingreso corriente hogar,2576,3082330.0,110684.85,51650.29,81865.70,134758.07,208928.66,284867.81,40.40
4,2018,CDMX entidad - ingreso corriente per cápita en...,7475,9219295.0,24247.95,8494.78,13457.47,24130.95,45923.58,77798.95,51.84
5,2020,CDMX entidad - ingreso corriente per cápita en...,9056,9260329.0,20032.87,8587.65,13841.17,23151.81,40109.40,53387.49,43.98
6,2022,CDMX entidad - ingreso corriente per cápita en...,8421,9346899.0,29875.83,11975.40,18320.08,31704.61,56044.56,87705.26,47.99
7,2024,CDMX entidad - ingreso corriente per cápita en...,8182,9381255.0,37063.14,14820.48,23986.94,40986.88,77707.01,107827.86,46.23
8,2018,ZM Valle de México - ingreso corriente per cáp...,13128,20520047.0,18808.45,7510.18,11720.90,20169.22,33281.55,51200.90,47.68
9,2020,ZM Valle de México - ingreso corriente per cáp...,16667,22113791.0,16975.95,7106.55,11331.88,18874.75,33017.46,49168.22,45.16


![CDMX entidad vs Zona Metropolitana del Valle de México, 2024](../reports/figures_documentacion/cdmx_vs_zmvm_2024.png)

WindowsPath('C:/Users/lucia/OneDrive/Escritorio/Fer/inegi-income-modeling/reports/figures_documentacion/cdmx_vs_zmvm_2024.png')

## Comparación entre grandes zonas metropolitanas

Para ingreso corriente per cápita del hogar se usa como estimando principal la distribución **entre personas**: cada fila de `mart_persona` hereda el ingreso per cápita de su hogar y se pondera con `factor`. El cálculo desde hogares con `factor * tot_integ` queda documentado como contraste conceptual, no como salida principal. Para ingreso laboral individual positivo también se usa la base de personas y `factor`.

In [22]:
met_pc = weighted_summary_custom(persona_zm, ["anio", "zona_metropolitana"], INC_PC_PERSONA, "factor")
met_lab = weighted_summary_custom(persona_zm, ["anio", "zona_metropolitana"], INC_LAB, "factor", positive_only=True)

display(Markdown("### Ingreso corriente per cápita del hogar distribuido entre personas, 2024"))
display(met_pc[met_pc["anio"].eq(2024)][["zona_metropolitana", "n_muestral", "poblacion_expandida", "media_ponderada", "p25", "mediana_ponderada", "p75", "p90", "p95", "gini_ponderado_0_100"]].sort_values("mediana_ponderada", ascending=False).round(2))

display(Markdown("### Ingreso laboral individual positivo, 2024"))
display(met_lab[met_lab["anio"].eq(2024)][["zona_metropolitana", "n_muestral", "poblacion_expandida", "p25", "mediana_ponderada", "p75", "p90"]].sort_values("mediana_ponderada", ascending=False).round(2))

### Ingreso corriente per cápita del hogar distribuido entre personas, 2024

,zona_metropolitana,n_muestral,poblacion_expandida,media_ponderada,p25,mediana_ponderada,p75,p90,p95,gini_ponderado_0_100
10,Monterrey,8264,5743578.0,37030.28,16232.80,24911.37,39539.08,61986.59,83253.93,45.35
9,Guadalajara,4487,6021378.0,28378.16,13886.48,21078.29,32467.96,51882.53,71413.04,39.31
11,Valle de México,14755,22419780.0,29006.21,13032.67,19704.90,32217.28,55445.79,81746.05,43.84


### Ingreso laboral individual positivo, 2024

,zona_metropolitana,n_muestral,poblacion_expandida,p25,mediana_ponderada,p75,p90
10,Monterrey,4073,2880616.0,24786.88,36684.76,54752.75,86703.11
9,Guadalajara,2361,3162965.0,19035.88,30412.52,46229.49,70923.91
11,Valle de México,7560,11422652.0,16508.15,26413.04,43036.85,72956.98


## Figuras territoriales 2024

Estas figuras usan ingreso corriente per cápita del hogar distribuido **entre personas**: una fila por persona en `mart_persona` y ponderador `factor`. La intención es visualizar brechas territoriales dentro de 2024, no comparar poder adquisitivo entre años.

In [23]:
regional_pc_doc = weighted_summary_custom(persona_geo, ["anio", "region_banxico"], INC_PC_PERSONA, "factor")
tam_loc_doc = weighted_summary_custom(persona_geo, ["anio", "tam_loc_desc"], INC_PC_PERSONA, "factor")
est_socio_doc = weighted_summary_custom(persona_geo, ["anio", "est_socio_desc"], INC_PC_PERSONA, "factor")

region_2024 = regional_pc_doc[regional_pc_doc["anio"].eq(2024)].copy().sort_values("mediana_ponderada")
fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(region_2024))
ax.hlines(y, region_2024["p25"], region_2024["p75"], color="#7a7a7a", linewidth=5, alpha=0.55, label="P25-P75")
ax.scatter(region_2024["mediana_ponderada"], y, color="#006d77", s=70, label="Mediana ponderada")
ax.set_yticks(y, region_2024["region_banxico"])
ax.set_title("Distribución regional del ingreso corriente per cápita, 2024")
ax.set_xlabel("Pesos nominales trimestrales")
ax.set_ylabel("Región Banxico")
ax.legend(loc="lower right")
ax.text(0.01, -0.16, "Ingreso per cápita del hogar distribuido entre personas; filas persona + factor.", transform=ax.transAxes, fontsize=9)
save_doc_figure(fig, "ingreso_pc_region_2024.png", "Distribución regional del ingreso corriente per cápita, 2024")

tam_2024 = tam_loc_doc[tam_loc_doc["anio"].eq(2024)].sort_values("mediana_ponderada")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(tam_2024["tam_loc_desc"], tam_2024["mediana_ponderada"], color=["#8d99ae", "#83c5be", "#e9c46a", "#006d77"])
ax.set_title("Gradiente por tamaño de localidad, 2024")
ax.set_xlabel("Mediana ponderada, pesos nominales trimestrales")
ax.set_ylabel("Tamaño de localidad")
ax.text(0.01, -0.20, "Ingreso corriente per cápita del hogar distribuido entre personas; filas persona + factor.", transform=ax.transAxes, fontsize=9)
save_doc_figure(fig, "gradiente_tam_loc_2024.png", "Gradiente por tamaño de localidad, 2024")

socio_order = ["Bajo", "Medio bajo", "Medio alto", "Alto"]
socio_2024 = est_socio_doc[est_socio_doc["anio"].eq(2024)].copy()
socio_2024["est_socio_desc"] = pd.Categorical(socio_2024["est_socio_desc"], categories=socio_order, ordered=True)
socio_2024 = socio_2024.sort_values("est_socio_desc")
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(socio_2024["est_socio_desc"].astype(str), socio_2024["mediana_ponderada"], marker="o", linewidth=2.5, color="#c1121f")
ax.set_title("Gradiente por estrato socioeconómico, 2024")
ax.set_xlabel("Estrato socioeconómico INEGI")
ax.set_ylabel("Mediana ponderada, pesos nominales trimestrales")
ax.text(0.01, -0.18, "Ingreso corriente per cápita del hogar distribuido entre personas; filas persona + factor.", transform=ax.transAxes, fontsize=9)
save_doc_figure(fig, "gradiente_est_socio_2024.png", "Gradiente por estrato socioeconómico, 2024")

display(Markdown("### Región Banxico, 2024"))
display(region_2024[["region_banxico", "n_muestral", "poblacion_expandida", "media_ponderada", "p25", "mediana_ponderada", "p75", "p90", "gini_ponderado_0_100"]].round(2))

![Distribución regional del ingreso corriente per cápita, 2024](../reports/figures_documentacion/ingreso_pc_region_2024.png)

![Gradiente por tamaño de localidad, 2024](../reports/figures_documentacion/gradiente_tam_loc_2024.png)

![Gradiente por estrato socioeconómico, 2024](../reports/figures_documentacion/gradiente_est_socio_2024.png)

### Región Banxico, 2024

,region_banxico,n_muestral,poblacion_expandida,media_ponderada,p25,mediana_ponderada,p75,p90,gini_ponderado_0_100
15,Sur,69804,29619793.0,16481.84,7125.62,12018.70,19712.02,31845.19,43.40
12,Centro,77400,49254755.0,23813.22,10648.70,16484.98,26802.98,44483.03,42.98
13,Centro Norte,91164,27597413.0,23829.08,11376.92,17585.13,27751.53,44167.28,40.53
14,Norte,70230,23854008.0,30149.80,14676.19,22268.90,34672.13,54061.12,40.06


## Grandes urbes vs contexto territorial desfavorecido

No se usa la palabra “marginado” porque aún no se incorporó CONAPO. El grupo desfavorecido se define solo con variables actuales: localidad menor de 2,500 habitantes y `est_socio_desc = Bajo`. Para evitar duplicidades, primero se asignan las tres zonas metropolitanas y luego, fuera de ellas, el grupo desfavorecido. El ingreso per cápita se reporta como distribución entre personas con `mart_persona` + `factor`.

In [24]:
persona_geo["grupo_brecha_territorial"] = pd.NA
mask_metro_persona = persona_geo["zona_metropolitana"].isin(zonas_prioritarias)
persona_geo.loc[mask_metro_persona, "grupo_brecha_territorial"] = persona_geo.loc[mask_metro_persona, "zona_metropolitana"]
mask_desfavorecido_persona = persona_geo["grupo_brecha_territorial"].isna() & persona_geo["tam_loc_desc"].eq("Localidades con menos de 2 500 habitantes") & persona_geo["est_socio_desc"].eq("Bajo")
persona_geo.loc[mask_desfavorecido_persona, "grupo_brecha_territorial"] = "Localidad pequeña y estrato socioeconómico bajo"

brecha_2024 = weighted_summary_custom(persona_geo[persona_geo["anio"].eq(2024) & persona_geo["grupo_brecha_territorial"].notna()], ["grupo_brecha_territorial"], INC_PC_PERSONA, "factor")
brecha_2024["p90_p10"] = brecha_2024["p90"] / brecha_2024["p10"]
brecha_2024["p75_p25"] = brecha_2024["p75"] / brecha_2024["p25"]
brecha_2024 = brecha_2024.sort_values("mediana_ponderada", ascending=False)
display(brecha_2024[["grupo_brecha_territorial", "n_muestral", "poblacion_expandida", "media_ponderada", "p10", "p25", "mediana_ponderada", "p75", "p90", "p90_p10", "p75_p25", "gini_ponderado_0_100"]].round(2))

hogar_geo["grupo_brecha_territorial_prev"] = pd.NA
mask_metro_hogar = hogar_geo["zona_metropolitana"].isin(zonas_prioritarias)
hogar_geo.loc[mask_metro_hogar, "grupo_brecha_territorial_prev"] = hogar_geo.loc[mask_metro_hogar, "zona_metropolitana"]
mask_desfavorecido_hogar = hogar_geo["grupo_brecha_territorial_prev"].isna() & hogar_geo["tam_loc_desc"].eq("Localidades con menos de 2 500 habitantes") & hogar_geo["est_socio_desc"].eq("Bajo")
hogar_geo.loc[mask_desfavorecido_hogar, "grupo_brecha_territorial_prev"] = "Localidad pequeña y estrato socioeconómico bajo"

brecha_2024_hogar_x_integrantes = weighted_summary_custom(hogar_geo[hogar_geo["anio"].eq(2024) & hogar_geo["grupo_brecha_territorial_prev"].notna()], ["grupo_brecha_territorial_prev"], INC_PC, W_PERSONAS_DESDE_HOGAR)
comparacion_correccion_brecha_2024 = brecha_2024_hogar_x_integrantes[["grupo_brecha_territorial_prev", "n_muestral", "poblacion_expandida", "mediana_ponderada", "gini_ponderado_0_100"]].merge(
    brecha_2024[["grupo_brecha_territorial", "n_muestral", "poblacion_expandida", "mediana_ponderada", "gini_ponderado_0_100"]],
    left_on="grupo_brecha_territorial_prev", right_on="grupo_brecha_territorial",
    suffixes=("_hogar_x_integrantes", "_persona_factor"),
)
comparacion_correccion_brecha_2024["dif_mediana"] = comparacion_correccion_brecha_2024["mediana_ponderada_persona_factor"] - comparacion_correccion_brecha_2024["mediana_ponderada_hogar_x_integrantes"]
comparacion_correccion_brecha_2024["dif_gini"] = comparacion_correccion_brecha_2024["gini_ponderado_0_100_persona_factor"] - comparacion_correccion_brecha_2024["gini_ponderado_0_100_hogar_x_integrantes"]
display(Markdown("### Corrección del estimando per cápita, 2024"))
display(comparacion_correccion_brecha_2024.drop(columns=["grupo_brecha_territorial"]).round(2))

hi = brecha_2024.iloc[0]
lo = brecha_2024.iloc[-1]
display(Markdown(f"**Brecha 2024:** la mediana ponderada de {hi['grupo_brecha_territorial']} es {hi['mediana_ponderada'] / lo['mediana_ponderada']:.2f} veces la de {lo['grupo_brecha_territorial']} para ingreso per cápita distribuido entre personas."))

fig, ax = plt.subplots(figsize=(9, 5))
plot = brecha_2024.sort_values("mediana_ponderada")
colors = ["#8d99ae" if "Localidad" in group else "#006d77" for group in plot["grupo_brecha_territorial"]]
ax.barh(plot["grupo_brecha_territorial"], plot["mediana_ponderada"], color=colors)
ax.set_title("Brecha territorial: mediana de ingreso per cápita, 2024")
ax.set_xlabel("Mediana ponderada, pesos nominales trimestrales")
ax.set_ylabel("Grupo territorial")
ax.text(0.01, -0.20, "Ingreso corriente per cápita del hogar distribuido entre personas; filas persona + factor.", transform=ax.transAxes, fontsize=9)
save_doc_figure(fig, "brecha_metropolitana_2024.png", "Brecha territorial por zonas metropolitanas y contexto desfavorecido, 2024")

,grupo_brecha_territorial,n_muestral,poblacion_expandida,media_ponderada,p10,p25,mediana_ponderada,p75,p90,p90_p10,p75_p25,gini_ponderado_0_100
2,Monterrey,8264,5743578.0,37030.28,10831.26,16232.80,24911.37,39539.08,61986.59,5.72,2.44,45.35
0,Guadalajara,4487,6021378.0,28378.16,9933.25,13886.48,21078.29,32467.96,51882.53,5.22,2.34,39.31
3,Valle de México,14755,22419780.0,29006.21,8737.02,13032.67,19704.90,32217.28,55445.79,6.35,2.47,43.84
1,Localidad pequeña y estrato socioeconómico bajo,50657,13981703.0,10152.95,3006.73,4887.97,7888.31,12355.48,18445.36,6.13,2.53,39.45


### Corrección del estimando per cápita, 2024

,grupo_brecha_territorial_prev,n_muestral_hogar_x_integrantes,poblacion_expandida_hogar_x_integrantes,mediana_ponderada_hogar_x_integrantes,gini_ponderado_0_100_hogar_x_integrantes,n_muestral_persona_factor,poblacion_expandida_persona_factor,mediana_ponderada_persona_factor,gini_ponderado_0_100_persona_factor,dif_mediana,dif_gini
0,Guadalajara,1373,6017359.0,21063.96,39.25,4487,6021378.0,21078.29,39.31,14.33,0.07
1,Localidad pequeña y estrato socioeconómico bajo,13724,13980966.0,7887.92,39.45,50657,13981703.0,7888.31,39.45,0.39,0.00
2,Monterrey,2509,5736479.0,24899.99,44.80,8264,5743578.0,24911.37,45.35,11.38,0.55
3,Valle de México,4540,22370536.0,19671.28,43.43,14755,22419780.0,19704.90,43.84,33.62,0.41


**Brecha 2024:** la mediana ponderada de Monterrey es 3.16 veces la de Localidad pequeña y estrato socioeconómico bajo para ingreso per cápita distribuido entre personas.

![Brecha territorial por zonas metropolitanas y contexto desfavorecido, 2024](../reports/figures_documentacion/brecha_metropolitana_2024.png)

WindowsPath('C:/Users/lucia/OneDrive/Escritorio/Fer/inegi-income-modeling/reports/figures_documentacion/brecha_metropolitana_2024.png')

## Figuras documentales generadas

Estas son las figuras seleccionadas para `reports/documentacion_final_en_desarrollo.md`.


In [25]:
figuras_documentacion = sorted(p.relative_to(ROOT).as_posix() for p in DOC_FIG_DIR.glob("*.png"))
display(pd.DataFrame({"figura": figuras_documentacion}))


,figura
0,reports/figures_documentacion/brecha_metropoli...
1,reports/figures_documentacion/cdmx_vs_zmvm_202...
2,reports/figures_documentacion/gini_banxico_com...
3,reports/figures_documentacion/gini_regiones_20...
4,reports/figures_documentacion/gradiente_est_so...
5,reports/figures_documentacion/gradiente_tam_lo...
6,reports/figures_documentacion/ingreso_pc_regio...


## Resumen de hallazgos actualizado

- `factor` es el factor de expansión oficial. En filas de hogar se usa `factor`; en filas de persona se usa `factor`. No se multiplica `factor * tot_integ` cuando cada integrante ya tiene una fila.
- En 2018 y 2020 `poblacion.csv` no trae `factor` directo; en 2022 y 2024 sí lo trae. El mart de personas conserva un `factor` por persona y permite estimar población con `sum(factor)`.
- `factor * tot_integ` solo se conserva para estimar personas desde una fila por hogar o para contrastar conceptualmente el ingreso per cápita distribuido entre personas desde hogares.
- El Gini nacional del ingreso corriente total del hogar ponderado por `factor` es 43.83 en 2018, 42.60 en 2020, 41.27 en 2022 y 40.06 en 2024.
- El Gini calculado desde `mart_hogar` y desde `concentradohogar` directo es idéntico hasta cuatro decimales; la construcción del mart no explica la discrepancia con Banxico.
- Frente a Banxico, el cálculo hogar-total queda abajo hasta 3.22 puntos. La variante diagnóstica de ingreso per cápita distribuido entre personas, calculada con `mart_persona` + `factor`, tiene discrepancia máxima cercana a 1.71 puntos.
- Para 2024, Monterrey tiene la mayor mediana ponderada de ingreso corriente per cápita distribuido entre personas entre las tres zonas metropolitanas revisadas: $24,911. Le siguen Guadalajara ($21,078) y Valle de México ($19,705).
- CDMX entidad no equivale a la Zona Metropolitana del Valle de México: en 2024 la mediana per cápita entre personas de CDMX entidad es $23,987 y la de la ZM Valle de México es $19,705.
- La brecha 2024 entre Monterrey y el grupo “localidad pequeña y estrato socioeconómico bajo” es de 3.16 veces en mediana ponderada de ingreso corriente per cápita distribuido entre personas.
- Todo sigue en pesos nominales trimestrales. Para comparar niveles monetarios entre años, el siguiente paso es deflactar; para Gini dentro de año, un deflactor común no cambia el índice.